# Code Lab - An Entire RAG Pipeline

En esta sección vamos a crear un **RAG pipeline** completo y desde cero, que servirá como base para profundizar en los capítulos posteriores. En este capítulo se incluirá un pipeline con las siguientes características:

- Vincular un LLM con una cuenta de OpenAI
- Instalación de paquetes de Python
- Web crawling, división de documentos y embedding chunks para la indexación de datos
- Búsqueda por vectores similares (vector similarity search)
- Generar respuestas integrando contexto a los prompts
- Sin interfaz 

En primer lugar instalamos e importamos las librerías necesarias:

In [1]:
%pip install langchain_community langchain_experimental langchain-ollama langchainhub chromadb langchain beautifulsoup4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 12.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 14.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 10.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 7.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.9/73.9 kB 14.0 MB/s eta 0:00:00
  Using cached certifi-2025.8.3-py3-none-any.whl.metadata (2.4 kB)
  Using cached idna-3.10-py3-none-any.whl.metadata (10 kB)
  Using cached charset_normalizer-3.4.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (36 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 29.4 MB/s eta 0:00:000:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
import os
from langchain_community.document_loaders import WebBaseLoader
import bs4
import ollama
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain import hub
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import chromadb
from langchain_community.vectorstores import Chroma
from langchain_experimental.text_splitter import SemanticChunker

USER_AGENT environment variable not set, consider setting it to identify your requests.


---

### **1. Indexing**

Ahora viene la primera fase del RAG, **indexing**, donde obtendremos los datos del prompt del usuario, les haremos un pre-procesamiento y los vectorizaremos. En este pequeño apartado haremos lo siguiente:

- Web loading y web crawling
- Dividir (splitting) los datos en chunks para que el algoritmo de vectorización de *Chroma* sea más eficiente
- Convertir los chunks en embeddings
- Añadir los chunks y embeddings a la *vector store* de *Chroma*

#### 1.1 Web Loading y Web Crawling

El contenido lo vamos a sacar de la siguiente página:

In [17]:
webPage = 'https://lilianweng.github.io/posts/2023-06-23-agent/'

loader = WebBaseLoader(
    web_path = webPage,
    bs_kwargs = dict(
        parse_only = bs4.SoupStrainer(
            class_ = ('post-title', 'post-header', 'post-content')
        )
    ),
)

docs = loader.load()

for doc in docs:
    print(doc)


page_content='

      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.
Agent System Overview#
In a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:

Planning

Subgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.
Reflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, thereby improving the quality of final results.


Memory

Short-t

De esta manera podemos obtener el contenido de una web como documentos.

*WebBaseLoader* hace mucho trabajo:

1. Petición HTTP a la URL que hemos especificado
2. Hace un parseo del HTML son *BeautifulSoup*, parseando únicamente los elementos incluidos en *parse_only*
3. Extrae el texto del parseo
4. Crea objetos tipo *document* con el contenido de la web que hemos extraído (que se guardan en la variable docs en este caso)

Una vez hecho esto pasamos al siguiente paso, splitting

#### 1.2 Splitting

En este paso simplemente vamos a dividir el documento que hemos obtenido anteriormente en distintos *chunks*, para reducir tiempo de procesamiento, convertiéndolos en textos mucho más manejables sin perder la coherencia de cada *chunk*. 

En nuestro caso vamos a usar *SemanticChunker* aunque hay otras muchas opciones:

In [41]:
embeddings = OllamaEmbeddings(model='nomic-embed-text')
text_splitter = SemanticChunker(embeddings)
splits = text_splitter.split_documents(docs)
print(len(splits))

21


Vemos que *SemanticChunker* nos divide el documento en 21 chunks. Esto se hace en función del contexto de cada chunk, en lugar de proporcionar una longitud establecida. Esto en general nos hace una mejor división del texto al ser semántica (por ello necesita un modelo como *OllamaEmbeddings*), pero es más pesado computacionalmente.

Vamos a probar a ver un Chunker diferente, el cual si se basa en tamaño concreto, conocido como *RecursiveCharacterTextSplitter*, uno de los splitters más usados de langchain.

In [40]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter_size = RecursiveCharacterTextSplitter(
    chunk_size = 500,   # Tamaño máximo de cada chunk
    chunk_overlap = 50,   # Indica cuantos caracteres se repiten entre un chunk y el siguiente (para no perder contexto)
    separators = ['\n\n', '\n', ' ', '']   # Lista ordenada de los separadores (intenta separar por parrafos, si no cabe en chunk_size prueba por lineas, etc.)
)

splits_size = text_splitter.split_documents(docs)
print(len(splits_size))

134


Y vemos que el tiempo en este caso es mucho menor, ya que estamos indicando el tamaño de los chunks y la separación es mucho más simple, pero la calidad de los chunks baja considerablemente. 

Hemos que decidir entre calidad o eficiencia, aunque en este caso, vamos a optar por calidad ya que no hay mucho texto.

#### 1.3 Embeddings y Vector Store de Chroma

A continuación vamos a crear la *vector store* con *Chroma* y vamos a guardar los *embeddings* de nuestro texto. Esto lo podemos hacer muy sencillamente con *Chroma* en *Python*:

In [ ]:
vector_store = Chroma.from_documents(
    documents = splits,
    embedding = OllamaEmbeddings(model='nomic-embed-text')
)

retriever = vector_store.as_retriever()

Internamente, el método *Chroma.from_documents()* está haciendo lo siguiente:

1. Itera sobre cada *Document* en la variable *splits*
2. Para cada *Document*, usa el embedding, en este caso *OllamaEmbeddings()* para generar el vector
3. Guarda el texto original y su correspondiente vector en la *vector store* de *Chroma*

Podemos ver que los embeddings se estan generando correctamente con el siguiente código (no forma dentro del código final):

In [72]:
data = vector_store.get(include=['embeddings'])

# Para ver los embeddings del primer chunk, no los imprimimos todos (hay 768 xd)
print(data['embeddings'][0][:30])

[ 0.02725698  0.05026644 -0.13049266 -0.07393885  0.03244552 -0.00505756
  0.02465732 -0.00350863 -0.02056746 -0.0068397  -0.01521997 -0.0046853
  0.10491869  0.03440397 -0.01472879  0.01154074  0.01415378 -0.07126766
 -0.00486376  0.01106874  0.03104531 -0.01692409  0.02466148 -0.01964626
  0.01312184  0.00217164  0.01849631 -0.04251862 -0.01740811 -0.00742486]


El *retriever* que se vio anteriormente, se utilizará para el algoritmo de *vector similarity* en nuestra *vector store*, ya que nos proporciona los métodos necesarios para ello.

Podemos ver un ejemplo de su uso, que nuevamente no forma parte del código oficial:

In [81]:
query = 'How does RAG compare with fine-tuning?'
relevant_docs = retriever.get_relevant_documents(query)
relevant_docs

[Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='Observation: ...'),
 Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='4. ... 5. ... Constraints:\n1. ~4000 word limit for short term memory. Your short term memory is short, so immediately save important information to files. 2. If you are unsure how you previously did something or want to recall past events, thinking about similar events will help you remember. 3.'),
 Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='2023). In both experiments on knowledge-intensive tasks and decision-making tasks, ReAct works better than the Act-only baseline where Thought: … step is removed. Reflexion (Shinn & Labash 2023) is a framework to equip agents with dynamic memory and self-reflection capabilities to improve reasoning skills. Reflexion has a standard RL setup, in which the reward model provide

Y este resultado es la información más relevante de nuestra *vector store* que se asemeja más al prompt. Sin embargo, esto es simplemente un ejemplo muy sencillo, ya que no hemos dado una respuesta, solo la información que es similar. Para ello pasamos a la siguiente sección...

---

### **2. Retriever y Generation**